# Flight Delay Prediction: ML Modeling (Final, Production-Oriented)

This notebook trains and evaluates machine learning models using the **leakage-safe modeling datasets**
produced by:

- `data_cleaning_feature_engineering.ipynb`

## Inputs
From `outputs/`:
- `flight_delay_train_features.parquet`
- `flight_delay_test_features.parquet`

## Target
- `IS_DELAYED` (arrival delay >= 15 minutes)
- `IS_DELAYED_15` is treated as an alias where present

## Models
1. Logistic Regression (baseline, interpretable)
2. XGBoost 
3. CatBoost 

## Evaluation
Primary metric: **PR-AUC** (appropriate for imbalanced "delay" class)  
Secondary metrics: ROC-AUC, Brier score

In [6]:
import os
import json
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.metrics import confusion_matrix
from sklearn.calibration import CalibratedClassifierCV


## 1) Load leakage-safe train/test feature tables


In [7]:
TRAIN_PATH = "/Users/nikitha/Documents/flight-delay-prediction-ice/outputs/flight_delay_train_features.parquet"
TEST_PATH  = "/Users/nikitha/Documents/flight-delay-prediction-ice/outputs/flight_delay_test_features.parquet"

train_df = pd.read_parquet(TRAIN_PATH)
test_df  = pd.read_parquet(TEST_PATH)

print("Train shape:", train_df.shape, "Test shape:", test_df.shape)
train_df.head()

Train shape: (118388, 70) Test shape: (29597, 70)


,YEAR,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,FL_DATE,UNIQUE_CARRIER,ORIGIN,ORIGIN_CITY_NAME,ORIGIN_STATE_ABR,...,ORIGIN_RISK,DEST_RISK,UNIQUE_CARRIER_RISK,ROUTE_RISK,CARRIER_ROUTE_RISK,TRAIN_ORIGIN_COUNT,TRAIN_DEST_COUNT,TRAIN_ROUTE_COUNT,TRAIN_UNIQUE_CARRIER_COUNT,TRAIN_CARRIER_ROUTE_COUNT
0,2017,2,5,1,1,2017-05-01,EV,ATL,"Atlanta, GA",GA,...,0.164977,0.198791,0.191210,0.198791,0.225986,58308,326,326,9148,148
1,2017,2,5,1,1,2017-05-01,DL,MLB,"Melbourne, FL",FL,...,0.106172,0.143650,0.130001,0.106172,0.106385,249,58354,249,77878,239
2,2017,2,5,1,1,2017-05-01,DL,TPA,"Tampa, FL",FL,...,0.174379,0.143650,0.130001,0.174379,0.126902,1084,58354,1084,77878,736
3,2017,2,5,1,1,2017-05-01,DL,GNV,"Gainesville, FL",FL,...,0.172097,0.143650,0.130001,0.172097,0.124940,210,58354,210,77878,28
4,2017,2,5,1,1,2017-05-01,DL,ATL,"Atlanta, GA",GA,...,0.164977,0.100682,0.130001,0.098811,0.116122,58308,603,585,77878,387


## 2) Define target, remove leakage columns, and select features

The modeling datasets were created to be leakage-safe, but still apply a final safeguard:
- Exclude `ARR_DELAY` if present (direct label leakage)
- Exclude raw `FL_DATE` from features (keep only for reporting)

Feature selection rationale:
- Using high-signal engineered features highlighted by EDA:
  - route/carrier/airport behavior via **risk encodings**
  - time-of-day and seasonal effects
  - distance/airtime characteristics
  - hub and traffic context as secondary signals

This avoids training on many redundant flags while keeping the strongest predictors.

In [8]:
# Target column
if "IS_DELAYED" in train_df.columns:
    TARGET = "IS_DELAYED"
elif "IS_DELAYED_15" in train_df.columns:
    TARGET = "IS_DELAYED_15"
else:
    raise KeyError("Target not found. Expected IS_DELAYED or IS_DELAYED_15.")

# Parse FL_DATE for reporting only
for df_ in (train_df, test_df):
    if "FL_DATE" in df_.columns:
        df_["FL_DATE"] = pd.to_datetime(df_["FL_DATE"], errors="coerce")

y_train = train_df[TARGET].astype(int).values
y_test  = test_df[TARGET].astype(int).values

print("Delay rate (train):", round(y_train.mean(), 4), "Delay rate (test):", round(y_test.mean(), 4))
if "FL_DATE" in train_df.columns:
    print("Train date range:", train_df["FL_DATE"].min(), "->", train_df["FL_DATE"].max())
if "FL_DATE" in test_df.columns:
    print("Test  date range:", test_df["FL_DATE"].min(), "->", test_df["FL_DATE"].max())

# Columns to always exclude from features (leakage safeguards)
ALWAYS_EXCLUDE = {TARGET, "IS_DELAYED_15", "ARR_DELAY", "FL_DATE"}
ALWAYS_EXCLUDE = {c for c in ALWAYS_EXCLUDE if c in train_df.columns}

# Candidate feature groups (use what exists)
CORE_TEMPORAL = [
    "MONTH", "DAY_OF_WEEK", "DAY_OF_MONTH", "QUARTER",
    "IS_WEEKEND", "SEASON", "IS_SUMMER", "IS_WINTER", "IS_HOLIDAY_SEASON",
    "DEP_HOUR_BIN"
]

FLIGHT_CHARACTERISTICS = [
    "DISTANCE", "AIR_TIME", "DISTANCE_GROUP", "DISTANCE_CAT", "DISTANCE_NORMALIZED",
    "IS_SHORT_HAUL", "IS_MEDIUM_HAUL", "IS_LONG_HAUL"
]

# Risk encodings created in the FE notebook (train-only, leakage-safe)
RISK_FEATURES = [c for c in train_df.columns if c.endswith("_RISK")]

# Train-only count features
COUNT_FEATURES = [c for c in train_df.columns if c.endswith("_COUNT")]

# Context / operational proxies
CONTEXT = [
    "IS_HUB_ORIGIN", "IS_HUB_DEST", "IS_HUB_TO_HUB",
    "IS_BUSY_ORIGIN", "IS_BUSY_DEST",
    "IS_POPULAR_ROUTE",
    "ROUTE_POPULARITY", "ORIGIN_TRAFFIC", "DEST_TRAFFIC", "CARRIER_VOLUME",
]

# Include raw categoricals for CatBoost; for LR/XGB they will be one-hot encoded.
RAW_CATEGORICALS = ["UNIQUE_CARRIER", "ORIGIN", "DEST", "ROUTE", "CARRIER_ROUTE",
                    "ORIGIN_STATE_ABR", "DEST_STATE_ABR"]

# Build selected feature list, keeping only those present
selected = []
for group in [CORE_TEMPORAL, FLIGHT_CHARACTERISTICS, CONTEXT, RAW_CATEGORICALS, RISK_FEATURES, COUNT_FEATURES]:
    for c in group:
        if c in train_df.columns and c not in ALWAYS_EXCLUDE:
            selected.append(c)

# De-duplicate while preserving order
selected = list(dict.fromkeys(selected))

# Final X matrices
X_train = train_df[selected].copy()
X_test  = test_df[selected].copy()

print("Selected features:", len(selected))
print("Risk features:", len(RISK_FEATURES), "Count features:", len(COUNT_FEATURES))
selected[:20]


Delay rate (train): 0.1549 Delay rate (test): 0.1291
Train date range: 2017-05-01 00:00:00 -> 2018-02-22 00:00:00
Test  date range: 2018-02-23 00:00:00 -> 2018-04-30 00:00:00
Selected features: 45
Risk features: 5 Count features: 5


['MONTH',
 'DAY_OF_WEEK',
 'DAY_OF_MONTH',
 'QUARTER',
 'IS_WEEKEND',
 'SEASON',
 'IS_SUMMER',
 'IS_WINTER',
 'IS_HOLIDAY_SEASON',
 'DEP_HOUR_BIN',
 'DISTANCE',
 'AIR_TIME',
 'DISTANCE_GROUP',
 'DISTANCE_CAT',
 'DISTANCE_NORMALIZED',
 'IS_SHORT_HAUL',
 'IS_MEDIUM_HAUL',
 'IS_LONG_HAUL',
 'IS_HUB_ORIGIN',
 'IS_HUB_DEST']

## 3) Preprocessing for Logistic Regression and XGBoost

- Numeric: median imputation + scaling (scaling is important for logistic regression)
- Categorical: most frequent imputation + one-hot encoding (sparse output)

CatBoost will bypass this and use the raw DataFrame with native categorical support.


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# Identify categorical vs numeric by dtype (object/category treated as categorical)
cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object" or str(X_train[c].dtype).startswith("category")]
num_cols = [c for c in X_train.columns if c not in cat_cols]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler(with_mean=False))
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=True))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),
    ],
    remainder="drop",
    sparse_threshold=1.0
)

print("Numeric features:", len(num_cols), "Categorical features:", len(cat_cols))
cat_cols


Numeric features: 35 Categorical features: 10


['SEASON',
 'DEP_HOUR_BIN',
 'DISTANCE_CAT',
 'UNIQUE_CARRIER',
 'ORIGIN',
 'DEST',
 'ROUTE',
 'CARRIER_ROUTE',
 'ORIGIN_STATE_ABR',
 'DEST_STATE_ABR']

## 4) Train models and compute metrics


In [10]:
def compute_metrics(y_true, p):
    return {
        "roc_auc": roc_auc_score(y_true, p),
        "pr_auc": average_precision_score(y_true, p),
        "brier": brier_score_loss(y_true, p),
    }

results = []
preds = {}  # store probabilities per model for later thresholding/calibration

# Logistic Regression baseline
lr = Pipeline(steps=[
    ("prep", preprocessor),
    ("clf", LogisticRegression(
        max_iter=7000,
        solver="saga",
        n_jobs=-1,
        random_state=42
    ))
])

lr.fit(X_train, y_train)
p_lr = lr.predict_proba(X_test)[:, 1]
preds["LogisticRegression"] = p_lr
m_lr = compute_metrics(y_test, p_lr)
results.append({"model": "LogisticRegression", **m_lr})
m_lr


{'roc_auc': 0.6716150525429482,
 'pr_auc': 0.27850518622502357,
 'brier': 0.10945741669829424}

In [11]:
# XGBoost (optional)
try:
    from xgboost import XGBClassifier
    xgb_available = True
except Exception as e:
    xgb_available = False
    print("XGBoost not available. Install with: pip install xgboost")
    print("Import error:", e)

xgb = None
if xgb_available:
    pos = int((y_train == 1).sum())
    neg = int((y_train == 0).sum())
    scale_pos_weight = neg / max(pos, 1)

    xgb = Pipeline(steps=[
        ("prep", preprocessor),
        ("clf", XGBClassifier(
            n_estimators=600,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            min_child_weight=1.0,
            scale_pos_weight=scale_pos_weight,
            objective="binary:logistic",
            eval_metric="aucpr",
            random_state=42,
            n_jobs=-1
        ))
    ])

    xgb.fit(X_train, y_train)
    p_xgb = xgb.predict_proba(X_test)[:, 1]
    preds["XGBoost"] = p_xgb
    m_xgb = compute_metrics(y_test, p_xgb)
    results.append({"model": "XGBoost", **m_xgb})
    m_xgb


In [12]:
# CatBoost (optional)
try:
    from catboost import CatBoostClassifier
    cb_available = True
except Exception as e:
    cb_available = False
    print("CatBoost not available. Install with: pip install catboost")
    print("Import error:", e)

cb = None
if cb_available:
    # CatBoost uses native categoricals; prepare a safe DataFrame
    X_train_cb = X_train.copy()
    X_test_cb  = X_test.copy()

    # Convert pandas missing to np.nan (CatBoost does not accept pandas <NA>)
    X_train_cb = X_train_cb.astype("object").where(~pd.isna(X_train_cb), np.nan)
    X_test_cb  = X_test_cb.astype("object").where(~pd.isna(X_test_cb), np.nan)

    # Identify categorical columns for CatBoost (object dtype)
    cat_cols_cb = [c for c in X_train_cb.columns if X_train_cb[c].dtype == "object"]
    cat_feature_indices = [X_train_cb.columns.get_loc(c) for c in cat_cols_cb]

    # Fill missing categoricals with token; keep as strings
    for c in cat_cols_cb:
        X_train_cb[c] = X_train_cb[c].astype(str)
        X_test_cb[c]  = X_test_cb[c].astype(str)
        X_train_cb.loc[X_train_cb[c].isin(["nan", "None", "<NA>"]), c] = "__MISSING__"
        X_test_cb.loc[X_test_cb[c].isin(["nan", "None", "<NA>"]), c] = "__MISSING__"

    # Ensure numeric columns are numeric floats
    for c in X_train_cb.columns:
        if c not in cat_cols_cb:
            X_train_cb[c] = pd.to_numeric(X_train_cb[c], errors="coerce").astype(float)
            X_test_cb[c]  = pd.to_numeric(X_test_cb[c], errors="coerce").astype(float)

    pos = int((y_train == 1).sum())
    neg = int((y_train == 0).sum())
    w1 = neg / max(pos, 1)

    cb = CatBoostClassifier(
        iterations=800,
        depth=6,
        learning_rate=0.05,
        loss_function="Logloss",
        eval_metric="PRAUC",
        class_weights=[1.0, float(w1)],
        random_seed=42,
        verbose=200
    )

    cb.fit(
        X_train_cb, y_train,
        cat_features=cat_feature_indices,
        eval_set=(X_test_cb, y_test),
        use_best_model=True
    )

    p_cb = cb.predict_proba(X_test_cb)[:, 1]
    preds["CatBoost"] = p_cb
    m_cb = compute_metrics(y_test, p_cb)
    results.append({"model": "CatBoost", **m_cb})
    m_cb


0:	learn: 0.7336665	test: 0.5996817	best: 0.5996817 (0)	total: 1.46s	remaining: 19m 26s
200:	learn: 0.8088429	test: 0.6592583	best: 0.6592583 (200)	total: 1m 53s	remaining: 5m 39s
400:	learn: 0.8202771	test: 0.6573833	best: 0.6601379 (247)	total: 3m 52s	remaining: 3m 51s
600:	learn: 0.8268629	test: 0.6584434	best: 0.6601379 (247)	total: 6m 6s	remaining: 2m 1s
799:	learn: 0.8325175	test: 0.6585752	best: 0.6601379 (247)	total: 8m 20s	remaining: 0us

bestTest = 0.6601379392
bestIteration = 247

Shrink model to first 248 iterations.


## 5) Model comparison


In [13]:
results_df = pd.DataFrame(results).sort_values("pr_auc", ascending=False).reset_index(drop=True)
results_df


,model,roc_auc,pr_auc,brier
0,CatBoost,0.693043,0.298819,0.186265
1,XGBoost,0.679085,0.290261,0.179715
2,LogisticRegression,0.671615,0.278505,0.109457


## 6) Threshold selection for product decisions

Default threshold 0.5 is rarely optimal for imbalanced problems.
This section produces a small table of precision/recall trade-offs.

We need to choose a threshold based on product requirements, for example:
- Target recall (catch more delays) if missed delays are costly for users
- Target precision (reduce false alerts) if alerts are disruptive

In [14]:
def threshold_table(y_true, p, thresholds):
    rows = []
    for thr in thresholds:
        y_pred = (p >= thr).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        rows.append({
            "threshold": float(thr),
            "precision": precision,
            "recall": recall,
            "tp": int(tp), "fp": int(fp), "fn": int(fn), "tn": int(tn)
        })
    return pd.DataFrame(rows)

best_model = results_df.loc[0, "model"]
p_best = preds[best_model]

thresholds = np.round(np.linspace(0.10, 0.90, 9), 2)
thr_df = threshold_table(y_test, p_best, thresholds)
thr_df


,threshold,precision,recall,tp,fp,fn,tn
0,0.1,0.129647,0.999477,3819,25638,2,138
1,0.2,0.133800,0.986391,3769,24400,52,1376
2,0.3,0.150856,0.898979,3435,19335,386,6441
3,0.4,0.195663,0.699032,2671,10980,1150,14796
4,0.5,0.254669,0.471081,1800,5268,2021,20508
5,0.6,0.353391,0.290500,1110,2031,2711,23745
6,0.7,0.463299,0.153625,587,680,3234,25096
7,0.8,0.580645,0.065951,252,182,3569,25594
8,0.9,0.824324,0.015964,61,13,3760,25763


## 7) Probability calibration

If app displays probabilities (e.g., "Delay risk: 35%"), calibrate the chosen model.
Calibration improves probability reliability (often lowers Brier score).

- For LogisticRegression/XGBoost (sklearn pipeline): `CalibratedClassifierCV`
- For CatBoost: isotonic regression over predicted probabilities (simple, effective)



In [15]:
from sklearn.isotonic import IsotonicRegression

calibrated = None
p_cal = None

if best_model in ["LogisticRegression", "XGBoost"]:
    base_model = lr if best_model == "LogisticRegression" else xgb
    calibrated = CalibratedClassifierCV(base_model, method="isotonic", cv="prefit")
    calibrated.fit(X_train, y_train)
    p_cal = calibrated.predict_proba(X_test)[:, 1]

elif best_model == "CatBoost" and cb is not None:
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(preds["CatBoost"], y_test)
    p_cal = iso.transform(preds["CatBoost"])
    calibrated = iso

if p_cal is not None:
    print("Uncalibrated:", compute_metrics(y_test, p_best))
    print("Calibrated:  ", compute_metrics(y_test, p_cal))
else:
    print("Calibration skipped (best model not available in this environment).")


Uncalibrated: {'roc_auc': 0.6930430801895046, 'pr_auc': 0.2988188111878081, 'brier': 0.18626527999466633}
Calibrated:   {'roc_auc': 0.6951479161925073, 'pr_auc': 0.2893199489043967, 'brier': 0.10311644763442854}


## 8) Export model artifacts

This section exports:
- Model artifact (joblib for sklearn pipelines, `.cbm` for CatBoost)
- `model_metadata.json` containing:
  - selected feature list
  - chosen operating threshold
  - test metrics
  - delay rates

Set `OPERATING_THRESHOLD` based on the threshold table above.


In [16]:
import joblib

ART_DIR = "model_artifacts_final"
os.makedirs(ART_DIR, exist_ok=True)

OPERATING_THRESHOLD = 0.30  # update based on thr_df for your chosen operating point

metadata = {
    "best_model": best_model,
    "selected_features": selected,
    "categorical_features_lr_xgb": cat_cols,
    "operating_threshold": OPERATING_THRESHOLD,
    "metrics_test": results_df.loc[0].to_dict(),
    "delay_rate_train": float(y_train.mean()),
    "delay_rate_test": float(y_test.mean()),
}

with open(os.path.join(ART_DIR, "model_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

if best_model == "LogisticRegression":
    joblib.dump(lr, os.path.join(ART_DIR, "best_model.joblib"))
    print("Saved sklearn model:", os.path.join(ART_DIR, "best_model.joblib"))

elif best_model == "XGBoost" and xgb is not None:
    joblib.dump(xgb, os.path.join(ART_DIR, "best_model.joblib"))
    print("Saved sklearn model:", os.path.join(ART_DIR, "best_model.joblib"))

elif best_model == "CatBoost" and cb is not None:
    cb.save_model(os.path.join(ART_DIR, "catboost_model.cbm"))
    print("Saved CatBoost model:", os.path.join(ART_DIR, "catboost_model.cbm"))

print("Saved metadata:", os.path.join(ART_DIR, "model_metadata.json"))
metadata


Saved CatBoost model: model_artifacts_final/catboost_model.cbm
Saved metadata: model_artifacts_final/model_metadata.json


{'best_model': 'CatBoost',
 'selected_features': ['MONTH',
  'DAY_OF_WEEK',
  'DAY_OF_MONTH',
  'QUARTER',
  'IS_WEEKEND',
  'SEASON',
  'IS_SUMMER',
  'IS_WINTER',
  'IS_HOLIDAY_SEASON',
  'DEP_HOUR_BIN',
  'DISTANCE',
  'AIR_TIME',
  'DISTANCE_GROUP',
  'DISTANCE_CAT',
  'DISTANCE_NORMALIZED',
  'IS_SHORT_HAUL',
  'IS_MEDIUM_HAUL',
  'IS_LONG_HAUL',
  'IS_HUB_ORIGIN',
  'IS_HUB_DEST',
  'IS_HUB_TO_HUB',
  'IS_BUSY_ORIGIN',
  'IS_BUSY_DEST',
  'IS_POPULAR_ROUTE',
  'ROUTE_POPULARITY',
  'ORIGIN_TRAFFIC',
  'DEST_TRAFFIC',
  'CARRIER_VOLUME',
  'UNIQUE_CARRIER',
  'ORIGIN',
  'DEST',
  'ROUTE',
  'CARRIER_ROUTE',
  'ORIGIN_STATE_ABR',
  'DEST_STATE_ABR',
  'ORIGIN_RISK',
  'DEST_RISK',
  'UNIQUE_CARRIER_RISK',
  'ROUTE_RISK',
  'CARRIER_ROUTE_RISK',
  'TRAIN_ORIGIN_COUNT',
  'TRAIN_DEST_COUNT',
  'TRAIN_ROUTE_COUNT',
  'TRAIN_UNIQUE_CARRIER_COUNT',
  'TRAIN_CARRIER_ROUTE_COUNT'],
 'categorical_features_lr_xgb': ['SEASON',
  'DEP_HOUR_BIN',
  'DISTANCE_CAT',
  'UNIQUE_CARRIER',
  'ORIGI

In [17]:
import json

# 1. Create a dictionary to store our feature lookups
artifacts = {
    "means": {},
    "mappings": {},
    "defaults": {}
}

# 2. Save Risk Encodings and Counts (using the train_df from your notebook)
# We map the raw value (e.g., 'ATL') to its Risk Score and Count
for col in ["ORIGIN", "DEST", "UNIQUE_CARRIER", "ROUTE", "CARRIER_ROUTE"]:
    # Risk Mapping
    if f"{col}_RISK" in train_df.columns:
        artifacts["mappings"][f"{col}_RISK"] = train_df.set_index(col)[f"{col}_RISK"].to_dict()
        artifacts["defaults"][f"{col}_RISK"] = train_df[f"{col}_RISK"].mean() # Fallback for unknown airports
        
    # Count Mapping
    if f"TRAIN_{col}_COUNT" in train_df.columns:
        artifacts["mappings"][f"TRAIN_{col}_COUNT"] = train_df.set_index(col)[f"TRAIN_{col}_COUNT"].to_dict()
        artifacts["defaults"][f"TRAIN_{col}_COUNT"] = 0

# 3. Save Standardization Stats (Mean/Std) for Distance
artifacts["means"]["DISTANCE"] = train_df["DISTANCE"].mean()
artifacts["means"]["DISTANCE_STD"] = train_df["DISTANCE"].std()

# 4. Save the artifacts to a JSON file
with open("model_artifacts_final/feature_lookups.json", "w") as f:
    json.dump(artifacts, f)

print("Feature lookup tables saved successfully!")

Feature lookup tables saved successfully!
